# Demo 3 · Pinned versus newest

**Slides:** "“latest” is a moving target" (slide 9), "Pin it, log it, cache it" (slide 15), demo step 03.

The same small evaluation runs against a pinned Gemini model and against a *floating* choice: code that
always picks the newest Flash model available today, the way a `-latest` alias or a "just use the newest" config does.
Every response records which model version answered, and responses are cached so a rerun makes no API calls.

Needs a Gemini API key stored as a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar).
Runtime: any. The free tier allows 5 requests per minute per model, so the first run is paced and takes
about 3 to 4 minutes. Reruns come from the cache and take a second. Run it before recording, not on stage.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q   # already cloned in this session: get the latest version
%cd /content/repo
!git log --oneline -1

In [ ]:
!pip install -q -U google-genai

## 1 · Connect and choose a pinned model

Model availability depends on your key, region and the day, so first **test** which Flash models answer. Then pin one of them explicitly in `PINNED_MODEL`.

In [ ]:
import re
from google import genai
from google.genai import types, errors
from google.colab import userdata

# A 60 s timeout per request, so a stuck call fails instead of hanging the notebook.
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"),
                      http_options=types.HttpOptions(timeout=60_000))
SKIP = ("image", "tts", "audio", "live", "embedding")   # other kinds of model (Flash-Lite is kept)

def version_key(name):
    return tuple(int(x) for x in re.findall(r"\d+", name))

candidates = sorted(
    (m.name.removeprefix("models/") for m in client.models.list()
     if "flash" in m.name and "generateContent" in (m.supported_actions or [])
     and not any(word in m.name for word in SKIP)),
    key=version_key, reverse=True)

AVAILABLE = []   # newest first
for name in candidates:
    try:
        client.models.generate_content(model=name, contents="Say OK.")
        AVAILABLE.append(name)
        print(f"OK          {name}")
    except errors.APIError as e:   # 404 not found, 403 no access, 429 no free quota, 503 overloaded
        print(f"unavailable {name}  ({e.code})")
print("\navailable:", AVAILABLE)

In [ ]:
# Pin an explicit, versioned model and change it only in a pull request.
# Leave it as None to pin automatically: the newest stable model that is not the newest overall.
# Once you see the list above, paste one ID here, e.g. PINNED_MODEL = "gemini-2.5-flash"
PINNED_MODEL = None

def is_stable(name):
    return not any(tag in name for tag in ("preview", "exp", "latest"))

def newest_flash():
    """The floating choice: whichever available Flash model is newest today."""
    return AVAILABLE[0]

assert AVAILABLE, "No Flash model answered: check the API key secret, or try again later."
FLOATING_MODEL = newest_flash()
if PINNED_MODEL is None:
    stable = [n for n in AVAILABLE if is_stable(n)]
    older = [n for n in stable if n != FLOATING_MODEL]
    PINNED_MODEL = (older or stable or AVAILABLE)[0]
assert PINNED_MODEL in AVAILABLE, f"{PINNED_MODEL} is not available today: pick one from {AVAILABLE}"

print("pinned:  ", PINNED_MODEL)
print("floating:", FLOATING_MODEL, "(newest Flash today; next month it may be another model)")

CONFIG = types.GenerateContentConfig(temperature=0, seed=1234)

## 2 · Ask with logging and a cache

The cache key is a hash of model, prompt and config. Changing any of them is a new request; repeating them is free and identical.

In [ ]:
import hashlib, json, pathlib, re, time
import httpx
from google.genai import errors

MIN_SECONDS_BETWEEN_CALLS = 13   # free tier: 5 requests per minute per model
_last_call = {}                   # per model, since the limit is per model

def generate(model, prompt):
    """Call the API, pacing requests per model and retrying rate limits, overloads and timeouts."""
    for attempt in range(4):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.monotonic() - _last_call.get(model, -1e9))
        if wait > 0:
            time.sleep(wait)
        _last_call[model] = time.monotonic()
        try:
            return client.models.generate_content(model=model, contents=prompt, config=CONFIG)
        except errors.ClientError as e:
            if e.code != 429:
                raise
            if "PerDay" in str(e):   # daily free quota used up: waiting will not help today
                raise RuntimeError(f"{model}: daily free quota used up. Pick another model from "
                                   f"AVAILABLE in section 1, or rerun tomorrow.") from None
            m = re.search(r"retry in ([0-9.]+)s", str(e))
            delay = min(float(m.group(1)) + 1 if m else 30, 65)
            print(f"  {model}: rate limited, waiting {delay:.0f} s ...", flush=True)
            time.sleep(delay)
        except (errors.ServerError, httpx.TimeoutException) as e:   # 503 overloaded, or no answer in 60 s
            delay = 15 * (attempt + 1)
            print(f"  {model}: no answer ({type(e).__name__}), retrying in {delay} s ...", flush=True)
            time.sleep(delay)
    raise RuntimeError(f"{model} kept failing: pick another model from AVAILABLE in section 1")

CACHE = pathlib.Path(".gemini_cache/responses.jsonl")
CACHE.parent.mkdir(exist_ok=True)
cache = {}
if CACHE.exists():
    for line in CACHE.read_text().splitlines():
        rec = json.loads(line)
        cache[rec["key"]] = rec

def ask(model: str, prompt: str) -> dict:
    config = CONFIG.model_dump(exclude_none=True)
    key = hashlib.sha256(json.dumps([model, prompt, config], sort_keys=True).encode()).hexdigest()
    if key in cache:
        return {**cache[key], "cached": True}
    response = generate(model, prompt)
    rec = {"key": key, "model": model, "model_version": response.model_version,
           "prompt": prompt, "config": config, "text": (response.text or "").strip()}
    cache[key] = rec
    with CACHE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    return {**rec, "cached": False}

## 3 · A tiny evaluation set

In [ ]:
EVAL = [
    ("The update fixed every crash I had reported.", "positive"),
    ("The battery now lasts half as long as before.", "negative"),
    ("The package arrived on Tuesday.", "neutral"),
    ("Support answered quickly but could not solve the problem.", "negative"),
    ("Setup took two minutes and everything just worked.", "positive"),
    ("The manual is available in three languages.", "neutral"),
    ("I would not buy this again.", "negative"),
    ("Honestly better than I expected for the price.", "positive"),
]
TEMPLATE = ("Classify the sentiment of this review. Answer with exactly one word: "
            "positive, negative or neutral.\n\nReview: {text}")

## 4 · Run against the pinned model and the floating one

The first run calls the API 16 times, paced for the free tier: about 2 minutes per model. Progress prints as it goes.

In [ ]:
import pandas as pd

def evaluate(model):
    rows = []
    for i, (text, expected) in enumerate(EVAL, 1):
        r = ask(model, TEMPLATE.format(text=text))
        print(f"{model}: {i}/{len(EVAL)}", "(cache)" if r["cached"] else "(API)", flush=True)
        answer = r["text"].lower().strip(" .")
        rows.append({"review": text[:40], "expected": expected, "answer": answer,
                     "correct": answer == expected, "model_version": r["model_version"],
                     "cached": r["cached"]})
    return pd.DataFrame(rows)

pinned = evaluate(PINNED_MODEL)
floating = evaluate(FLOATING_MODEL)
print(f"pinned   {PINNED_MODEL}: accuracy {pinned.correct.mean():.2f}, served by {set(pinned.model_version)}")
print(f"floating {FLOATING_MODEL}: accuracy {floating.correct.mean():.2f}, served by {set(floating.model_version)}")
pd.concat({"pinned": pinned, "floating": floating}, axis=1)

## 5 · Rerun: everything comes from the cache

Same answers, zero API calls. This is what makes an LLM evaluation reproducible next month.

In [ ]:
again = evaluate(PINNED_MODEL)
print("all cached:", again.cached.all(), "| identical answers:", (again.answer == pinned.answer).all())

## 6 · Log the evaluation

In [ ]:
!pip install -q mlflow
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("devfest-gemini-eval")
for label, model, df in [("pinned", PINNED_MODEL, pinned), ("floating", FLOATING_MODEL, floating)]:
    with mlflow.start_run(run_name=label):
        mlflow.log_params({"model": model, "model_version": ",".join(sorted(set(df.model_version))),
                           "temperature": 0, "seed": 1234, "n_items": len(df)})
        mlflow.log_metric("accuracy", float(df.correct.mean()))
        mlflow.log_artifact(str(CACHE))
print("logged")

## What to say

The two models may give the same answers today. The point is the record: when the floating choice moves,
the logged `model_version` shows it, and the cache lets you rerun last month's evaluation exactly.
Upgrade the pinned ID deliberately, in a pull request, with this evaluation rerun.